<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Report-Without-Ground-Truth-v1.0" data-toc-modified-id="Report-Without-Ground-Truth-v1.0-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Report Without Ground Truth v1.0</a></span><ul class="toc-item"><li><span><a href="#Input-prameters" data-toc-modified-id="Input-prameters-1.1"><span class="toc-item-num">1.1&nbsp;&nbsp;</span>Input prameters</a></span></li></ul></li><li><span><a href="#Variants-Statistics" data-toc-modified-id="Variants-Statistics-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Variants Statistics</a></span></li><li><span><a href="#Snp-statistics" data-toc-modified-id="Snp-statistics-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>Snp statistics</a></span><ul class="toc-item"><li><span><a href="#Trinucleotide-Heatmap" data-toc-modified-id="Trinucleotide-Heatmap-3.1"><span class="toc-item-num">3.1&nbsp;&nbsp;</span>Trinucleotide Heatmap</a></span></li></ul></li><li><span><a href="#INDEL-statistics" data-toc-modified-id="INDEL-statistics-4"><span class="toc-item-num">4&nbsp;&nbsp;</span>INDEL statistics</a></span><ul class="toc-item"><li><span><a href="#Distribution-of-indel-lengths" data-toc-modified-id="Distribution-of-indel-lengths-4.1"><span class="toc-item-num">4.1&nbsp;&nbsp;</span>Distribution of indel lengths</a></span></li><li><span><a href="#Insertion-Deletion-Statistics-for-h-mer-indels" data-toc-modified-id="Insertion-Deletion-Statistics-for-h-mer-indels-4.2"><span class="toc-item-num">4.2&nbsp;&nbsp;</span>Insertion Deletion Statistics for h-mer indels</a></span></li></ul></li></ul></div>

# Report Without Ground Truth v1.0
## Input prameters

In [ ]:
import json
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, Image, Markdown
from ugbio_core.vcfbed import vcftools

In [ ]:
run_id = "NA"
pipeline_version = "NA"
h5_statistics = "no_gt_statistics.h5"
h5_statistics_wgs = "no_gt_statistics_wgs.h5"
annotation_intervals_names = "annotation_intervals_names"  # comma-separated
is_somatic = False
filtered_vcf = "filtered_vcf.vcf.gz"
interval_list = ""
ref_fasta = "Homo_sapiens_assembly38.fasta"
ref_fasta_dict = "Homo_sapiens_assembly38.dict"
signature_plots_folder = ""
cosmic_signatures = "cosmic_signatures.json"
h5outfile = "no_gt_report.h5"

In [ ]:
warnings.filterwarnings("ignore")


SCIENTIFIC_NOTATION_THRESHOLD = 1e6
pd.options.display.float_format = lambda x: f"{x:,.1e}" if x > SCIENTIFIC_NOTATION_THRESHOLD else f"{x:,.1f}"
pd.options.display.max_colwidth = 100

prm = {
    "run_id": run_id,
    "pipeline_version": pipeline_version,
    "h5_statistics": h5_statistics,
    "h5_statistics_wgs": h5_statistics_wgs,
    "annotation_intervals_names": annotation_intervals_names,
    "is_somatic": is_somatic,
    "filtered_vcf": filtered_vcf,
    "interval_list": interval_list,
    "signature_plots_folder": signature_plots_folder,
    "cosmic_signatures": cosmic_signatures,
}
prm = {name: str(value) for name, value in prm.items()}

no_gt_statistics = h5_statistics
no_gt_statistics_wgs = h5_statistics_wgs
is_somatic = str(is_somatic).lower() == "true"
annotation_intervals_names = annotation_intervals_names.split(",")

prmdf = pd.DataFrame.from_dict(prm, orient="index", columns=["value"])
prmdf.to_hdf(h5outfile, key="parameters")
prmdf

In [ ]:
SMALL_SIZE = 12
MEDIUM_SIZE = 18
BIGGER_SIZE = 26
TITLE_SIZE = 36
FIGSIZE = (16, 8)
GRID = True
COLORS = [
    "blue",
    "red",
    "green",
    "magenta",
    "black",
    "brown",
    "orange",
    "salmon",
    "teal",
    "coral",
    "lime",
    "purple",
    "cyan",
    "lavender",
    "turquoise",
    "darkgreen",
    "tan",
    "lightblue",
    "pink",
    "yellow",
    "gold",
]

plt.rc("font", size=SMALL_SIZE)  # controls default text sizes
plt.rc("axes", titlesize=TITLE_SIZE)  # fontsize of the axes title
plt.rc("axes", labelsize=BIGGER_SIZE)  # fontsize of the x and y labels
plt.rc("axes", grid=GRID)  # is grid on
plt.rc("axes", prop_cycle=plt.cycler(color=COLORS))
plt.rc("xtick", labelsize=MEDIUM_SIZE)  # fontsize of the tick labels
plt.rc("ytick", labelsize=MEDIUM_SIZE)  # fontsize of the tick labels
plt.rc("legend", fontsize=MEDIUM_SIZE)  # legend fontsize
plt.rc("figure", titlesize=TITLE_SIZE)  # fontsize of the figure title
plt.rc("figure", figsize=FIGSIZE)  # size of the figure

HTML("""
<style>
.output_png {
    display: table-cell;
    text-align: center;
    vertical-align: middle;
}
</style>
""")

# Variants Statistics

In [ ]:
# GATK VariantEval table and type of every reported metric, in the order of the report
METRIC_SOURCES = {
    "nEvalVariants": ("CompOverlap", int),
    "nDeletions": ("CountVariants", int),
    "nInsertions": ("CountVariants", int),
    "nSNPs": ("CountVariants", int),
    "nTi": ("TiTvVariantEvaluator", int),
    "nTv": ("TiTvVariantEvaluator", int),
    "tiTvRatio": ("TiTvVariantEvaluator", float),
    "SNP_to_indel_ratio": ("IndelSummary", float),
    "indel_novelty_rate": ("IndelSummary", float),
    "insertion_to_deletion_ratio": ("IndelSummary", float),
    "nHets": ("CountVariants", int),
    "nHomVar": ("CountVariants", int),
    "nMultiSNPs": ("MultiallelicSummary", int),
    "nMultiIndels": ("MultiallelicSummary", int),
}
# Biallelic counts: all minus multiallelic, from MultiallelicSummary
BIALLELIC_METRICS = {"nBiallelicSNPs": ("nSNPs", "nMultiSNPs"), "nBiallelicIndels": ("nIndels", "nMultiIndels")}
NOVELTIES = ["novel", "known", "all"]


def metric_table_annotation(h5_path, annotation_name="none"):
    tables = {}

    def eval_value(table_name, metric, novelty, value_type):
        if table_name not in tables:
            tables[table_name] = pd.read_hdf(h5_path, f"eval_{table_name}")
        table = tables[table_name]
        selected = table.loc[(table["JexlExpression"] == annotation_name) & (table["Novelty"] == novelty), metric]
        return value_type(pd.to_numeric(selected, errors="coerce").iloc[0])

    metric_table = pd.DataFrame(index=[*METRIC_SOURCES, *BIALLELIC_METRICS], columns=NOVELTIES)
    for novelty in NOVELTIES:
        for metric, (table_name, value_type) in METRIC_SOURCES.items():
            metric_table.loc[metric, novelty] = eval_value(table_name, metric, novelty, value_type)
        for metric, (total, multi) in BIALLELIC_METRICS.items():
            metric_table.loc[metric, novelty] = eval_value("MultiallelicSummary", total, novelty, int) - eval_value(
                "MultiallelicSummary", multi, novelty, int
            )
    if annotation_name == "none":
        annotation_name = "wgs"
    metric_table.index = pd.MultiIndex.from_tuples(
        [(annotation_name, itm) for itm in metric_table.index], names=["regions", "metrics"]
    )
    return metric_table

In [ ]:
def metric_table(annotation_names, h5_path):
    frames = []
    for annotation_name in annotation_names:
        cur_result = metric_table_annotation(h5_path, annotation_name)
        frames.append(cur_result)
    result = pd.concat(frames)
    return result, frames


merged_df, frames = metric_table(["none"] + annotation_intervals_names, no_gt_statistics)
merged_df.to_hdf(h5outfile, key="variants_statistics")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

In [ ]:
def multi_column_df_display(list_dfs, cols=2):
    html_table = "<table style='width:100%; border:0px'>{content}</table>"
    html_row = "<tr style='border:0px'>{content}</tr>"
    html_cell = (
        "<td style='width:{width}%;vertical-align:top;border:1px solid !important;background-color: white'>"
        "{{content}}</td>"
    )
    html_cell = html_cell.format(width=100 / cols)
    cells = [html_cell.format(content=df.to_html()) for df in list_dfs]
    cells += (len(list_dfs) % cols) * [html_cell.format(content="")]  # pad
    rows = [html_row.format(content="".join(cells[i : i + cols])) for i in range(0, len(cells), cols)]
    display(HTML(html_table.format(content="".join(rows))))


multi_column_df_display(frames)


<ul>
<li><b>nEvalVariants</b>- the number of variants <br></li>
<li><b>nDeletions</b>- the number of variants determined to be deletions <br></li>
<li><b>nInsertions</b>- the number of variants determined to be insertions <br></li>
<li><b>nSNPs</b>- the number of variants determined to be single-nucleotide polymorphisms <br></li>
<li><b>nTi</b>- number of transition variants (A↔G or T↔C) <br></li>
<li><b>nTv</b>- number of transversion variants (A↔T or G↔C) <br></li>
<li><b>SNP_to_indel_ratio</b>- n_SNPs divided by n_indels <br></li>
<li><b>indel_novelty_rate</b>- n_novel_indels divided by n_indels<br></li>
<li><b>insertion_to_deletion_ratio</b>- n_insertions divided by n_deletions<br></li>
<li><b>nHets</b>- the number of heterozygous loci<br></li>   
<li><b>nHomVar</b>- the number of homozygous variant loci<br></li>    
<li><b>nMultiSNPs</b>- number of multiallelic SNPs<br></li>
<li><b>nMultiIndels</b>- number of multiallelic indels<br></li>
<li><b>nBiallelicSNPs</b>- nSNPs minus nMultiSNPs<br></li>
<li><b>nBiallelicIndels</b>- nIndels minus nMultiIndels<br></li>
</ul>





In [ ]:
with pd.HDFStore(no_gt_statistics, "r") as store:
    has_callable_size = "/callable_size" in store.keys()
if has_callable_size:
    # written by run_no_gt_report full_analysis as a pandas Series
    callable_size = int(pd.read_hdf(no_gt_statistics, "callable_size").iloc[0])
    display(Markdown("""# Callable interval\n"""))
    display(Markdown(f"Callable interval size: {callable_size:,}\n"))
    s = pd.DataFrame(pd.Series(callable_size, index=["bp"], name="callable_size"))
    s.to_hdf(h5outfile, key="callable_size")

In [ ]:
if is_somatic:
    display(
        Markdown("""# Distribution of Allele fractions\n
    """)
    )
    display(Markdown("""Histogram of 100 bins of Allele fraction for the somatic data."""))

In [ ]:
def af_hist_graph(filtered_af_hist, annotation_name="none"):
    nbins = 100
    bin_edges = pd.Series(np.arange(0, 1, 1 / nbins))

    for group in filtered_af_hist.columns:
        plt.plot(
            bin_edges,
            filtered_af_hist[group] / np.sum(filtered_af_hist[group]) * 100,
            alpha=0.5,
            label=group,
            linewidth="3",
        )
        plt.legend(loc="upper right")
        plt.xlabel("Estimated allele fraction")
        plt.ylabel("% Variants")
        plt.title(f"{annotation_name} Pass filter AF histogram", fontsize=24)
    plt.show()

In [ ]:
if is_somatic:
    filtered_af_hist = pd.read_hdf(no_gt_statistics_wgs, "af_hist")
    af_hist_graph(filtered_af_hist, "wgs")

In [ ]:
# Read vcf to df
MAX_VARIANTS_TO_PLOT = 100000
if is_somatic:
    # convert vcf to df
    vcf_df = vcftools.get_vcf_df(filtered_vcf)
    # randomly remove rows if the vcf is too large
    if vcf_df.shape[0] > MAX_VARIANTS_TO_PLOT:
        vcf_df = vcf_df.sample(n=MAX_VARIANTS_TO_PLOT, random_state=np.random.default_rng()).sort_index()

    if "vaf" in vcf_df.columns:
        vcf_df["af"] = vcf_df["vaf"]

    vcf_df["af"] = vcf_df["af"].apply(lambda x: x[0] if isinstance(x, tuple) else x)

In [ ]:
def af_along_genome(vcf_df):
    # for having different color for each chromosome
    vcf_df["ind"] = range(len(vcf_df))
    df_grouped = vcf_df.groupby("chrom")
    fig = plt.figure(figsize=(16, 10))  # Set the figure size
    ax = fig.add_subplot(111)

    colors = ["#281c65", "#622097", "#a0178a", "#eb2c5d", "#fc6840", "#f8c93c"]
    # for locating the x-labels of the different chr
    x_labels = []
    x_labels_pos = []
    for num, (name, group) in enumerate(df_grouped):
        group.plot(kind="scatter", x="ind", y="af", color=colors[num % len(colors)], ax=ax, alpha=0.2)
        x_labels.append(name)
        x_labels_pos.append(group["ind"].iloc[0])
    ax.set_xticks(x_labels_pos)
    ax.set_xticklabels(x_labels, rotation="vertical")

    # set axis limits
    ax.set_xlim([0, len(vcf_df)])
    ax.set_ylim([0, 1])

    # x axis label
    ax.set_xlabel("Chromosome")
    ax.set_ylabel("Allele Frequency")

    # show the graph
    plt.show()

In [ ]:
if is_somatic:
    display(
        Markdown("""# Variant allele frequency along genome positions\n
    """)
    )
    display(
        Markdown("""Allele frequency values of variants along the genome. Each variant is a point in the graph
    and each color is another chromosome. """)
    )
    af_along_genome(vcf_df)

In [ ]:
def af_vs_coverage(vcf_df):
    fig = plt.figure(figsize=(16, 10))  # Set the figure size
    ax = fig.add_subplot(111)

    ax.scatter(vcf_df["dp"], vcf_df["af"], alpha=0.05)

    # set axis limits
    ax.set_ylim([0, 1])

    # x axis label
    ax.set_xlabel("Depth")
    ax.set_ylabel("Allele Frequency")

    # show the graph
    plt.show()

In [ ]:
if is_somatic:
    display(
        Markdown("""# Variant allele frequency Vs. Depth\n
    """)
    )
    display(
        Markdown(
            "Allele frequency values of variants with their corresponding depth. Each variant is a point in the graph"
        )
    )
    af_vs_coverage(vcf_df)

# Snp statistics
Histogram for motif alterations

In [ ]:
base_colors = {"A": "b", "C": "r", "T": "y", "G": "g"}
dpi = 150


def plot_motifs0_bars(motifs_0, ax=None, title_prefix=""):
    if ax is None:
        plt.figure(figsize=(3, 6))
        ax = plt.gca()
    plt.sca(ax)

    x = (motifs_0).sort_index(ascending=False)

    bbox_extra_artists = [plt.title(f"{title_prefix} {x.sum():.1E} SNPs".replace("E+0", "E"), fontsize=17)]
    (x / motifs_0.sum() * 100).plot.barh(
        color=[base_colors.get(v) for v in x.index.get_level_values("ref")],
        ax=ax,
    )
    plt.yticks(
        ticks=range(6),
        labels=[f"{ref}->{alt}" for ref, alt in x.index],
    )
    for j, (_, row) in enumerate(x.to_frame().iterrows()):
        bbox_extra_artists += [
            plt.text(
                row["size"] / motifs_0.sum() * 101,
                j + 0.03,
                f"{row['size']:.1E}".replace("E+0", "E"),
            )
        ]
    plt.ylabel("")
    plt.xlabel("Frequency [%]", fontsize=18)
    # plt.xlim(0, ax.get_xlim()[1]+0.1)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    return bbox_extra_artists

In [ ]:
def snps_statistics_annot(no_gt_statistics, annotation_name="none"):
    motifs = pd.read_hdf(no_gt_statistics, "snp_motifs")

    motifs_0 = motifs.reset_index()
    motifs_0 = (
        motifs_0.assign(ref=motifs_0["ref_motif"].str.slice(1, 2)).groupby(["ref", "alt_1"]).agg({"size": "sum"})
    )["size"]

    fig, ax = plt.subplots(figsize=(6, 6))
    plot_motifs0_bars(motifs_0, ax, title_prefix=annotation_name)

    fig.tight_layout()

In [ ]:
snps_statistics_annot(no_gt_statistics_wgs, "wgs")

In [ ]:
if is_somatic:
    display(
        Markdown("""# Trinucleotide Histogram \n
    """)
    )
    display(Markdown("""Histogram for motif alterations in trinucleotide context."""))

    # The sample profile plots are named after the sample in the SigProfiler input
    with pd.HDFStore(no_gt_statistics_wgs, "r") as store:
        has_sample_name = "/signatures_sample_name" in store.keys()
    if has_sample_name:
        vcf_prefix = pd.read_hdf(no_gt_statistics_wgs, "signatures_sample_name").iloc[0]
    else:
        vcf_prefix = os.path.basename(filtered_vcf).split(".")[0]
    with open(cosmic_signatures) as f:
        cosmic_signatures_file = json.load(f)

    def plot_table_and_pngs(key_to_read, png_prefix, key_to_save, profile_key_name):
        sig_activities = pd.read_hdf(no_gt_statistics_wgs, key=key_to_read)
        sig_activities.name = "activity"
        sig_activities = sig_activities.astype(float).astype(int)
        sig_activities = sig_activities.sort_values(ascending=False).to_frame()
        sig_activities["link"] = None
        sig_activities["description"] = None
        for sig_name in list(sig_activities.index):
            if sig_name in cosmic_signatures_file:
                sig_activities.loc[sig_name, "description"] = cosmic_signatures_file[sig_name]["descprition"]
                sig_activities.loc[sig_name, "link"] = cosmic_signatures_file[sig_name]["link"]
        # Save the table
        sig_activities.to_hdf(h5outfile, key=key_to_save)
        # Save the profile
        profile = pd.read_hdf(no_gt_statistics_wgs, profile_key_name)
        profile.index = profile.index.astype(str)
        profile.to_hdf(h5outfile, key=profile_key_name)
        # Generate the HTML table
        sig_activities["link"] = sig_activities["link"].apply(lambda x: f'<a href="{x}">{x}</a>')
        html_table = sig_activities.to_html(index=True, classes="my-table", justify="left", escape=False)
        styled_table = html_table.replace("<td>", "<td style='text-align: left;'>").replace(
            "<th>", "<th style='text-align: center;'>"
        )
        # Display the table
        display(HTML(styled_table))

        for sig_name in list(sig_activities.index):
            filename = os.path.join(signature_plots_folder, f"{png_prefix}{sig_name}.png")
            if sig_activities.loc[sig_name, "description"] is not None:
                display(Markdown(f'**{sig_name}**:{sig_activities.loc[sig_name,"description"]}'))
                display(HTML(f"{sig_activities.loc[sig_name,'link']}"))
            display(Image(filename, width=2000, height=900))

In [ ]:
if is_somatic and os.path.exists(os.path.join(signature_plots_folder, f"SBS_96_plots_{vcf_prefix}.png")):
    display(
        Markdown("""## SBS sample histogram\n
    """)
    )
    # load activity table
    display(Image(os.path.join(signature_plots_folder, f"SBS_96_plots_{vcf_prefix}.png"), width=2000, height=900))
    display(
        Markdown("""## Significant SBS Signatures\n
    """)
    )
    plot_table_and_pngs("SBS96_activity", "SBS_96_plots_", "sbs_activities", "SBS96_profile")

In [ ]:
if is_somatic and os.path.exists(os.path.join(signature_plots_folder, f"ID_83_plots_{vcf_prefix}.png")):
    display(
        Markdown("""## ID sample histogram\n
    """)
    )
    # load activity table
    display(Image(os.path.join(signature_plots_folder, f"ID_83_plots_{vcf_prefix}.png"), width=2000, height=900))
    display(
        Markdown("""## Significant ID Signatures\n
    """)
    )
    plot_table_and_pngs("ID83_activity", "ID_83_plots_", "id_activities", "ID83_profile")

In [ ]:
if is_somatic and os.path.exists(os.path.join(signature_plots_folder, f"DBS_78_plots_{vcf_prefix}.png")):
    display(
        Markdown("""## DBS sample histogram\n
    """)
    )
    # load activity table
    display(Image(os.path.join(signature_plots_folder, f"DBS_78_plots_{vcf_prefix}.png"), width=2000, height=900))
    display(
        Markdown("""## Significant DBS Signatures\n
    """)
    )
    plot_table_and_pngs("DBS78_activity", "DBS_78_plots_", "dbs_activities", "DBS78_profile")

# INDEL statistics

## Distribution of indel lengths

In [ ]:
def ins_del_hete_graph_annot(known_indel, novel_indel, title=None):
    positions = np.arange(20)
    fig = plt.figure()
    ax = fig.add_axes([0, 0, 1, 1])

    ax.bar(positions + 0.00, known_indel, color="r", width=0.20)
    ax.bar(positions + 0.20, novel_indel, color="mediumseagreen", width=0.20)

    ax.set_title(title)
    ax.set_xticks(np.arange(0, 20))
    ax.set_xticklabels([-10, -9, -8, -7, -6, -5, -4, -3, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
    ax.legend(labels=["known_indel", "novel_indel"])
    ax.set_xlabel("Indel Length")


def ins_del_hete_graph(no_gt_statistics):
    indel_length_histogram = pd.read_hdf(no_gt_statistics, "eval_IndelLengthHistogram")

    known_indel = indel_length_histogram[indel_length_histogram["Novelty"] == "known"]["Freq"].astype(float)
    novel_indel = indel_length_histogram[indel_length_histogram["Novelty"] == "novel"]["Freq"].astype(float)

    ins_del_hete_graph_annot(known_indel, novel_indel, title="wgs")


ins_del_hete_graph(no_gt_statistics_wgs)

## Insertion Deletion Statistics for h-mer indels
Number of 1-base insretion deletions in hmer of each length

In [ ]:
if not is_somatic:
    display(
        Markdown("""## Heterozygous
    """)
    )

In [ ]:
def ins_del_hete_graph(ins_del, ax, title):
    positions = np.arange(12)

    ax.bar(positions + 0.00, ins_del.iloc[0], color="r", width=0.3)
    ax.bar(positions + 0.3, ins_del.iloc[1], color="mediumseagreen", width=0.3)
    ax.bar(positions + 0.00, -ins_del.iloc[2], color="purple", width=0.3)
    ax.bar(positions + 0.3, -ins_del.iloc[3], color="g", width=0.3)

    ax.set_title(title, fontsize=18)
    ax.set_xticks(np.arange(0, 12))
    ax.set_xticklabels(pd.Series(np.arange(1, 13)).apply(lambda x: f"hmer {x}"), rotation=45)
    ax.legend(labels=["ins A", "ins G", "del A", "del G"])
    ax.set_yticks(ax.get_yticks())

    ax.set_yticklabels([abs(x).astype(int) for x in ax.get_yticks()])
    ax.set_ylabel("Deletion        /       Insertion", loc="center")

In [ ]:
annotation_name = "wgs"
ins_del_hete = pd.read_hdf(no_gt_statistics_wgs, "ins_del_hete")
ins_del_hete.columns = pd.Series(ins_del_hete.columns).apply(lambda x: f"h-mer {x}")

fig, ax = plt.subplots(figsize=(9, 9))
ins_del_hete_graph(ins_del_hete, ax, f"{annotation_name} #h-mer insertions / deletions")

In [ ]:
if not is_somatic:
    display(
        Markdown("""## Homozygous
    """)
    )

In [ ]:
if not is_somatic:
    ins_del_homo = pd.read_hdf(no_gt_statistics_wgs, "ins_del_homo")
    ins_del_homo.columns = pd.Series(ins_del_homo.columns).apply(lambda x: f"h-mer {x}")
    fig, ax = plt.subplots(figsize=(9, 9))
    ins_del_hete_graph(ins_del_homo, ax, "wgs # insertions / deletions")